In [ ]:
### Project: Ethereum Fraud Detection

This project focuses on detecting fraudulent Ethereum wallet activity using real-world transaction data. The goal is to help crypto platforms prioritize suspicious users for manual review using machine learning, especially under data imbalance and operational constraints.



In [33]:
!pip install xgboost
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load data
df = pd.read_csv("data/transaction_dataset.csv")

# Quick preview
df.head()

In [ ]:
df_clean = df.drop(columns=['Unnamed: 0', 'Index', 'Address'], errors='ignore')

non_numeric_cols = df_clean.select_dtypes(include='object').columns.tolist()
df_clean = df_clean.drop(columns=non_numeric_cols)

X = df_clean.drop(columns=['FLAG'], errors='ignore')
y = df_clean['FLAG']
X = X.fillna(0)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, solver='liblinear'),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        eval_metric='logloss',
        random_state=42
    )
}

model_scores = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, scoring='f1', cv=cv)
    model_scores[name] = scores
    print(f"{name} 每折 F1-score：{scores}")
    print(f"{name} 平均 F1-score：{scores.mean():.4f}\n")

avg_scores = {name: np.mean(score) for name, score in model_scores.items()}
score_df = pd.DataFrame.from_dict(avg_scores, orient='index', columns=['Avg F1-score'])
score_df = score_df.sort_values(by='Avg F1-score', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=score_df, x='Avg F1-score', y=score_df.index)
plt.title('Model Comparison (F1-score)')
plt.xlabel('F1-score (5-fold CV)')
plt.ylabel('Model')
plt.tight_layout()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

final_model = XGBClassifier(
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss',
    random_state=42
)
final_model.fit(X_train, y_train)

y_proba = final_model.predict_proba(X_test)[:, 1]

risk_df = X_test.copy()
risk_df['fraud_score'] = y_proba
risk_df['actual'] = y_test.values

top_ks = list(range(50, 1001, 50))  
precisions = []

for k in top_ks:
    topk = risk_df.sort_values(by='fraud_score', ascending=False).head(k)
    precision_k = (topk['actual'] == 1).mean()
    precisions.append(precision_k)
    print(f"🔍 Precision@Top {k}: {precision_k:.2%}")

plt.figure(figsize=(8,5))
plt.plot(top_ks, precisions, marker='o')
plt.title('📊 Precision@TopK - Fraud Detection')
plt.xlabel('Top K Wallets with Highest Fraud Score')
plt.ylabel('Precision (Fraud Rate)')
plt.grid(True)
plt.xticks(top_ks)
plt.ylim(0.6, 1.01)
plt.show()


In [ ]:
### Conclusion

This project shows that with a well-designed ML pipeline, it is possible to accurately detect fraud in Ethereum wallet activity using imbalanced, real-world data. Precision@TopK analysis reveals that our model can help fraud teams focus on the most suspicious users, saving time and improving effectiveness.

### Future Work

- Incorporate temporal patterns of transactions
- Add address network graph features (e.g. betweenness, connected wallets)
- Try anomaly detection methods for unseen fraud types (unsupervised learning)
